# Rolex Watch Price Analysis - Part 4: Statistical Analysis
## Final Project - Data Analytics

## Libraries and settings

In [ ]:
import os
import numpy as np
import pandas as pd
import scipy.stats as stats
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import f_oneway, chi2_contingency, pearsonr

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print(os.getcwd())

## Import Cleaned Data

In [ ]:
df = pd.read_csv('rolex_data_cleaned.csv', encoding='utf-8')

print(f"Dataset shape: {df.shape}")
df.head()

## 1. Correlation Analysis with P-Values

### 1.1 Price vs Age Correlation

In [ ]:
df_clean = df[['price', 'age']].dropna()

r, p_value = pearsonr(df_clean['age'], df_clean['price'])

plt.figure(figsize=(8, 5))
plt.scatter(df_clean['age'], df_clean['price'], s=10, alpha=0.5, color='#2E86AB')
plt.xlabel('Age (years)', fontsize=11)
plt.ylabel('Price (CHF)', fontsize=11)
plt.title(f'Price vs Age (r={r:.4f}, p={p_value:.4f})', fontsize=12, pad=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Pearson Correlation Analysis: Price vs Age")
print(f"  Correlation coefficient (r): {r:.4f}")
print(f"  p-value: {p_value:.6f}")
print(f"\nInterpretation:")
if p_value < 0.05:
    if r > 0:
        print(f"  There is a statistically significant positive correlation between age and price (p < 0.05).")
        print(f"  Older watches tend to have higher prices.")
    else:
        print(f"  There is a statistically significant negative correlation between age and price (p < 0.05).")
        print(f"  Older watches tend to have lower prices.")
else:
    print(f"  There is no statistically significant correlation between age and price (p >= 0.05).")

### 1.2 Price vs Year Correlation

In [ ]:
df_clean2 = df[['price', 'year']].dropna()

r2, p_value2 = pearsonr(df_clean2['year'], df_clean2['price'])

plt.figure(figsize=(8, 5))
plt.scatter(df_clean2['year'], df_clean2['price'], s=10, alpha=0.5, color='#F18F01')
plt.xlabel('Production Year', fontsize=11)
plt.ylabel('Price (CHF)', fontsize=11)
plt.title(f'Price vs Production Year (r={r2:.4f}, p={p_value2:.4f})', fontsize=12, pad=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Pearson Correlation Analysis: Price vs Production Year")
print(f"  Correlation coefficient (r): {r2:.4f}")
print(f"  p-value: {p_value2:.6f}")
print(f"\nInterpretation:")
if p_value2 < 0.05:
    if r2 > 0:
        print(f"  There is a statistically significant positive correlation between production year and price (p < 0.05).")
        print(f"  Newer watches tend to have higher prices.")
    else:
        print(f"  There is a statistically significant negative correlation between production year and price (p < 0.05).")
        print(f"  Newer watches tend to have lower prices.")
else:
    print(f"  There is no statistically significant correlation between production year and price (p >= 0.05).")

### 1.3 Correlation Matrix for Numerical Variables

In [ ]:
numerical_cols = ['price', 'year', 'age', 'has_box', 'has_papers', 'has_complete_set', 'is_professional']

corr_matrix = df[numerical_cols].corr()

def calculate_pvalues(df_in):
    df_temp = df_in.dropna()
    dfcols = pd.DataFrame(columns=df_temp.columns)
    pvalues = dfcols.transpose().join(dfcols, how='outer')
    for r in df_temp.columns:
        for c in df_temp.columns:
            if r != c:
                pvalues[r][c] = pearsonr(df_temp[r], df_temp[c])[1]
            else:
                pvalues[r][c] = 0
    return pvalues

pvalues = calculate_pvalues(df[numerical_cols])

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='RdBu_r', center=0, 
            square=True, linewidths=1, ax=axes[0], cbar_kws={"shrink": 0.8})
axes[0].set_title('Correlation Coefficients', fontsize=12, pad=10)

sns.heatmap(pvalues.astype(float), annot=True, fmt='.4f', cmap='RdYlGn_r', 
            square=True, linewidths=1, ax=axes[1], cbar_kws={"shrink": 0.8})
axes[1].set_title('P-Values', fontsize=12, pad=10)

plt.tight_layout()
plt.show()

print("\nCorrelation with Price (sorted by absolute value):")
price_corr = corr_matrix['price'].sort_values(key=abs, ascending=False)
for var in price_corr.index:
    if var != 'price':
        p_val = pvalues.loc[var, 'price']
        sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "n.s."
        print(f"  {var:20s}: r={price_corr[var]:7.4f}, p={p_val:.6f} {sig}")

### Interpretation

The correlation analysis reveals which numerical variables have statistically significant linear relationships with price. Variables with p-values < 0.05 show significant correlations. The strength of the relationship is indicated by the correlation coefficient (r), where values closer to 1 or -1 indicate stronger relationships.

## 2. ANOVA Test: Price Differences Across Conditions

### 2.1 One-Way ANOVA for Condition Categories

In [ ]:
condition_groups = [group['price'].values for name, group in df.groupby('condition_category') if len(group) >= 2]
condition_names = [name for name, group in df.groupby('condition_category') if len(group) >= 2]

f_statistic, p_value_anova = f_oneway(*condition_groups)

plt.figure(figsize=(12, 6))
df.boxplot(column='price', by='condition_category', figsize=(12, 6), patch_artist=True)
plt.suptitle('')
plt.title(f'Price Distribution by Condition Category\n(ANOVA F={f_statistic:.2f}, p={p_value_anova:.6f})', 
          fontsize=12, pad=10)
plt.xlabel('Condition Category', fontsize=11)
plt.ylabel('Price (CHF)', fontsize=11)
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"One-Way ANOVA: Price Differences Across Condition Categories")
print(f"  F-statistic: {f_statistic:.4f}")
print(f"  p-value: {p_value_anova:.6f}")
print(f"\nDescriptive Statistics by Condition:")
condition_stats = df.groupby('condition_category')['price'].agg(['mean', 'median', 'std', 'count'])
print(condition_stats.round(2))
print(f"\nInterpretation:")
if p_value_anova < 0.05:
    print(f"  There are statistically significant price differences across condition categories (p < 0.05).")
    print(f"  The condition of a watch significantly affects its price.")
    print(f"  At least one condition category has a significantly different mean price than the others.")
else:
    print(f"  There are no statistically significant price differences across condition categories (p >= 0.05).")
    print(f"  The condition does not significantly affect the price.")

### 2.2 ANOVA for Material Categories

In [ ]:
material_groups = [group['price'].values for name, group in df.groupby('material_category') if len(group) >= 2]
material_names = [name for name, group in df.groupby('material_category') if len(group) >= 2]

f_stat_material, p_val_material = f_oneway(*material_groups)

plt.figure(figsize=(12, 6))
df.boxplot(column='price', by='material_category', figsize=(12, 6), patch_artist=True)
plt.suptitle('')
plt.title(f'Price Distribution by Material Category\n(ANOVA F={f_stat_material:.2f}, p={p_val_material:.6f})', 
          fontsize=12, pad=10)
plt.xlabel('Material Category', fontsize=11)
plt.ylabel('Price (CHF)', fontsize=11)
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"One-Way ANOVA: Price Differences Across Material Categories")
print(f"  F-statistic: {f_stat_material:.4f}")
print(f"  p-value: {p_val_material:.6f}")
print(f"\nDescriptive Statistics by Material:")
material_stats = df.groupby('material_category')['price'].agg(['mean', 'median', 'std', 'count'])
print(material_stats.round(2))
print(f"\nInterpretation:")
if p_val_material < 0.05:
    print(f"  There are statistically significant price differences across material categories (p < 0.05).")
    print(f"  The material composition significantly affects watch prices.")
    print(f"  Precious metals command different prices than stainless steel.")
else:
    print(f"  There are no statistically significant price differences across material categories (p >= 0.05).")

### 2.3 ANOVA for Movement Types

In [ ]:
movement_groups = [group['price'].values for name, group in df.groupby('movement_type') if len(group) >= 2]

if len(movement_groups) >= 2:
    f_stat_movement, p_val_movement = f_oneway(*movement_groups)
    
    plt.figure(figsize=(10, 6))
    df.boxplot(column='price', by='movement_type', figsize=(10, 6), patch_artist=True)
    plt.suptitle('')
    plt.title(f'Price Distribution by Movement Type\n(ANOVA F={f_stat_movement:.2f}, p={p_val_movement:.6f})', 
              fontsize=12, pad=10)
    plt.xlabel('Movement Type', fontsize=11)
    plt.ylabel('Price (CHF)', fontsize=11)
    plt.xticks(rotation=45)
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(f"One-Way ANOVA: Price Differences Across Movement Types")
    print(f"  F-statistic: {f_stat_movement:.4f}")
    print(f"  p-value: {p_val_movement:.6f}")
    print(f"\nDescriptive Statistics by Movement Type:")
    movement_stats = df.groupby('movement_type')['price'].agg(['mean', 'median', 'std', 'count'])
    print(movement_stats.round(2))
    print(f"\nInterpretation:")
    if p_val_movement < 0.05:
        print(f"  There are statistically significant price differences across movement types (p < 0.05).")
    else:
        print(f"  There are no statistically significant price differences across movement types (p >= 0.05).")
else:
    print("Not enough movement type groups for ANOVA analysis.")

### Interpretation

The ANOVA tests reveal whether there are statistically significant price differences across different categorical groups. A p-value < 0.05 indicates that at least one group has a significantly different mean price than the others.

## 3. Chi-Squared Test for Categorical Associations

### 3.1 Chi-Squared Test: Condition vs Material

In [ ]:
contingency_table = pd.crosstab(df['condition_category'], df['material_category'])

chi2, p_value_chi, dof, expected = chi2_contingency(contingency_table)

print(f"Chi-Squared Test: Condition vs Material Category")
print(f"  Chi-squared statistic: {chi2:.4f}")
print(f"  p-value: {p_value_chi:.6f}")
print(f"  Degrees of freedom: {dof}")
print(f"\nContingency Table:")
print(contingency_table)
print(f"\nInterpretation:")
if p_value_chi < 0.05:
    print(f"  There is a statistically significant association between condition and material (p < 0.05).")
    print(f"  The distribution of materials differs significantly across condition categories.")
else:
    print(f"  There is no statistically significant association between condition and material (p >= 0.05).")
    print(f"  Condition and material are independent.")

plt.figure(figsize=(12, 6))
contingency_pct = contingency_table.div(contingency_table.sum(axis=1), axis=0) * 100
contingency_pct.plot(kind='bar', stacked=False, figsize=(12, 6), alpha=0.8)
plt.title(f'Distribution of Materials by Condition Category\n(Chi-squared={chi2:.2f}, p={p_value_chi:.6f})', 
          fontsize=12, pad=10)
plt.xlabel('Condition Category', fontsize=11)
plt.ylabel('Percentage', fontsize=11)
plt.xticks(rotation=45)
plt.legend(title='Material', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### 3.2 Chi-Squared Test: Condition vs Seller Type

In [ ]:
contingency_table2 = pd.crosstab(df['condition_category'], df['seller_type'])

chi2_2, p_value_chi2, dof2, expected2 = chi2_contingency(contingency_table2)

print(f"Chi-Squared Test: Condition vs Seller Type")
print(f"  Chi-squared statistic: {chi2_2:.4f}")
print(f"  p-value: {p_value_chi2:.6f}")
print(f"  Degrees of freedom: {dof2}")
print(f"\nContingency Table:")
print(contingency_table2)
print(f"\nInterpretation:")
if p_value_chi2 < 0.05:
    print(f"  There is a statistically significant association between condition and seller type (p < 0.05).")
    print(f"  Professional dealers and private sellers have different distributions of watch conditions.")
else:
    print(f"  There is no statistically significant association between condition and seller type (p >= 0.05).")
    print(f"  Condition and seller type are independent.")

plt.figure(figsize=(10, 6))
contingency_pct2 = contingency_table2.div(contingency_table2.sum(axis=1), axis=0) * 100
contingency_pct2.plot(kind='bar', stacked=False, figsize=(10, 6), alpha=0.8)
plt.title(f'Distribution of Seller Types by Condition Category\n(Chi-squared={chi2_2:.2f}, p={p_value_chi2:.6f})', 
          fontsize=12, pad=10)
plt.xlabel('Condition Category', fontsize=11)
plt.ylabel('Percentage', fontsize=11)
plt.xticks(rotation=45)
plt.legend(title='Seller Type', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### 3.3 Chi-Squared Test: Material vs Movement Type

In [ ]:
contingency_table3 = pd.crosstab(df['material_category'], df['movement_type'])

chi2_3, p_value_chi3, dof3, expected3 = chi2_contingency(contingency_table3)

print(f"Chi-Squared Test: Material vs Movement Type")
print(f"  Chi-squared statistic: {chi2_3:.4f}")
print(f"  p-value: {p_value_chi3:.6f}")
print(f"  Degrees of freedom: {dof3}")
print(f"\nContingency Table:")
print(contingency_table3)
print(f"\nInterpretation:")
if p_value_chi3 < 0.05:
    print(f"  There is a statistically significant association between material and movement type (p < 0.05).")
    print(f"  Different materials are associated with different movement types.")
else:
    print(f"  There is no statistically significant association between material and movement type (p >= 0.05).")
    print(f"  Material and movement type are independent.")

### 3.4 Chi-Squared Test: Has Complete Set vs Seller Type

In [ ]:
df['has_complete_set_label'] = df['has_complete_set'].map({0: 'No Complete Set', 1: 'Complete Set'})
contingency_table4 = pd.crosstab(df['has_complete_set_label'], df['seller_type'])

chi2_4, p_value_chi4, dof4, expected4 = chi2_contingency(contingency_table4)

print(f"Chi-Squared Test: Complete Documentation vs Seller Type")
print(f"  Chi-squared statistic: {chi2_4:.4f}")
print(f"  p-value: {p_value_chi4:.6f}")
print(f"  Degrees of freedom: {dof4}")
print(f"\nContingency Table:")
print(contingency_table4)
print(f"\nInterpretation:")
if p_value_chi4 < 0.05:
    print(f"  There is a statistically significant association between complete documentation and seller type (p < 0.05).")
    print(f"  Professional dealers and private sellers differ in the proportion of complete sets they offer.")
else:
    print(f"  There is no statistically significant association between complete documentation and seller type (p >= 0.05).")

plt.figure(figsize=(8, 5))
contingency_pct4 = contingency_table4.div(contingency_table4.sum(axis=1), axis=0) * 100
contingency_pct4.plot(kind='bar', stacked=False, figsize=(8, 5), alpha=0.8)
plt.title(f'Distribution of Seller Types by Documentation Status\n(Chi-squared={chi2_4:.2f}, p={p_value_chi4:.6f})', 
          fontsize=12, pad=10)
plt.xlabel('Documentation Status', fontsize=11)
plt.ylabel('Percentage', fontsize=11)
plt.xticks(rotation=0)
plt.legend(title='Seller Type')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### Interpretation

Chi-squared tests assess whether there are statistically significant associations between categorical variables. A p-value < 0.05 indicates that the two variables are not independent - they have a significant relationship.

## Summary

This statistical analysis provided rigorous testing of relationships in the Rolex watch data:

### Correlation Analysis
- Examined linear relationships between numerical variables and price
- Calculated p-values to determine statistical significance
- Identified which numerical factors have significant correlations with price

### ANOVA Tests
- Tested whether mean prices differ significantly across categorical groups
- Examined condition categories, material types, and movement types
- Confirmed which categorical variables have statistically significant effects on price

### Chi-Squared Tests
- Assessed associations between pairs of categorical variables
- Determined whether distributions are independent or related
- Revealed structural relationships in the data (e.g., whether certain materials are more common in specific conditions)

All tests include p-values and clear interpretations, providing a solid statistical foundation for subsequent modeling work.

## Jupyter notebook --footer info-- (please always provide this at the end of each submitted notebook)

In [ ]:
import os
import platform
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')